## Semantic Similarity

In [1]:
import nltk
import pandas as pd

In [ ]:
for resource in [
    "punkt_tab",
    "averaged_perceptron_tagger_eng",
    "wordnet",
]:
    nltk.download(resource, quiet=True)

In [2]:
from scipy.stats import pearsonr
from similarity_scores import noun_verb_wup

def wup_correlation(file_path, name):
    df = pd.read_csv(file_path)
    
    if "wup" not in df.columns:
        df["wup"] = df.apply(lambda row: noun_verb_wup(row["S1"], row["S2"]), axis=1)
        df.to_csv(file_path, index=False)

    r, _ = pearsonr(df["wup"], df["score"])
    print(f"{name:<8}\tPearson correlation: {r:.4f}")


wup_correlation("data/stss131.csv", "STSS-131")
wup_correlation("data/sts12.csv", "STS-2012")
wup_correlation("data/sts13.csv", "STS-2013")
wup_correlation("data/sts14.csv", "STS-2014")
wup_correlation("data/sts15.csv", "STS-2015")
wup_correlation("data/sts16.csv", "STS-2016")
wup_correlation("data/stsb.csv", "STSB")
wup_correlation("data/sickr.csv", "SICK-R")
wup_correlation("data/mrpc.csv", "MRPC")

STSS-131	Pearson correlation: 0.3824
STS-2012	Pearson correlation: 0.1751
STS-2013	Pearson correlation: 0.2818
STS-2014	Pearson correlation: 0.3136
STS-2015	Pearson correlation: 0.3570
STS-2016	Pearson correlation: 0.2093
STSB    	Pearson correlation: 0.2428
SICK-R  	Pearson correlation: 0.4445
MRPC    	Pearson correlation: 0.1759


In [3]:
import gensim.downloader as api

w2v_model = api.load("word2vec-google-news-300")

In [3]:
from similarity_scores import sentence_similarity

def w2v_correlation(file_path, name):
    df = pd.read_csv(file_path)
    
    if "w2v" not in df.columns:
        df["w2v"] = df.apply(lambda row: sentence_similarity(w2v_model, row["S1"], row["S2"]), axis=1)
        df.to_csv(file_path, index=False)

    r, _ = pearsonr(df["w2v"], df["score"])
    print(f"{name:<8}\tPearson correlation: {r:.4f}")
    
    
w2v_correlation("data/stss131.csv", "STSS-131")
w2v_correlation("data/sts12.csv", "STS-2012")
w2v_correlation("data/sts13.csv", "STS-2013")
w2v_correlation("data/sts14.csv", "STS-2014")
w2v_correlation("data/sts15.csv", "STS-2015")
w2v_correlation("data/sts16.csv", "STS-2016")
w2v_correlation("data/stsb.csv", "STSB")
w2v_correlation("data/sickr.csv", "SICK-R")
w2v_correlation("data/mrpc.csv", "MRPC")

STSS-131	Pearson correlation: 0.7588
STS-2012	Pearson correlation: 0.4317
STS-2013	Pearson correlation: 0.6711
STS-2014	Pearson correlation: 0.6380
STS-2015	Pearson correlation: 0.7002
STS-2016	Pearson correlation: 0.5860
STSB    	Pearson correlation: 0.6274
SICK-R  	Pearson correlation: 0.7006
MRPC    	Pearson correlation: 0.3968


In [ ]:
from sentence_transformers import SentenceTransformer

ft_model = api.load("fasttext-wiki-news-subwords-300")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
roberta_model = SentenceTransformer("stsb-roberta-base-v2")
albert_model = SentenceTransformer("paraphrase-albert-small-v2")

In [11]:
from similarity_scores import bert_similarity

def evaluate_similarity_models(file_path, file_name):
    df = pd.read_csv(file_path)

    similarity_models = {
        "wup": lambda s1, s2: noun_verb_wup(s1, s2),
        "w2v": lambda s1, s2: sentence_similarity(w2v_model, s1, s2),
        "ft": lambda s1, s2: sentence_similarity(ft_model, s1, s2),
        "sbert": lambda s1, s2: bert_similarity(sbert_model, s1, s2),
        "roberta": lambda s1, s2: bert_similarity(roberta_model, s1, s2),
        "albert": lambda s1, s2: bert_similarity(albert_model, s1, s2),
    }

    for name, fn in similarity_models.items():
        if name not in df.columns:
            df[name] = df.apply(lambda row: fn(row["S1"], row["S2"]), axis=1)

    df.to_csv(file_path, index=False)
    
    wup_r, _ = pearsonr(df["wup"], df["score"])
    w2v_r, _ = pearsonr(df["w2v"], df["score"])
    ft_r, _ = pearsonr(df["ft"], df["score"])
    sbert_r, _ = pearsonr(df["sbert"], df["score"])
    roberta_r, _ = pearsonr(df["roberta"], df["score"])
    albert_r, _ = pearsonr(df["albert"], df["score"])
    
    print(
        f"{file_name:<8}Wu & Palmer: {wup_r:.4f} | Word2vec: {w2v_r:.4f}" +
        f" | FastText: {ft_r:.4f} | SBERT: {sbert_r:.4f}" + 
        f" | RoBERTa: {roberta_r:.4f} | ALBERT: {albert_r:.4f}"
    )
  

print("Pearson correlation coefficients")
evaluate_similarity_models("data/stss131.csv", "STSS-131")
evaluate_similarity_models("data/sts12.csv", "STS-2012")
evaluate_similarity_models("data/sts13.csv", "STS-2013")
evaluate_similarity_models("data/sts14.csv", "STS-2014")
evaluate_similarity_models("data/sts15.csv", "STS-2015")
evaluate_similarity_models("data/sts16.csv", "STS-2016")
evaluate_similarity_models("data/stsb.csv", "STSB")
evaluate_similarity_models("data/sickr.csv", "SICK-R")
evaluate_similarity_models("data/mrpc.csv", "MRPC")
    

Pearson correlation coefficients
STSS-131Wu & Palmer: 0.3824 | Word2vec: 0.7588 | FastText: 0.4914 | SBERT: 0.9180 | RoBERTa: 0.9048 | ALBERT: 0.9054
STS-2012Wu & Palmer: 0.1751 | Word2vec: 0.4317 | FastText: 0.2867 | SBERT: 0.6163 | RoBERTa: 0.7867 | ALBERT: 0.6236
STS-2013Wu & Palmer: 0.2818 | Word2vec: 0.6711 | FastText: 0.2855 | SBERT: 0.8015 | RoBERTa: 0.9067 | ALBERT: 0.8115
STS-2014Wu & Palmer: 0.3136 | Word2vec: 0.6380 | FastText: 0.2000 | SBERT: 0.8080 | RoBERTa: 0.9061 | ALBERT: 0.8074
STS-2015Wu & Palmer: 0.3570 | Word2vec: 0.7002 | FastText: 0.3592 | SBERT: 0.8474 | RoBERTa: 0.8988 | ALBERT: 0.8406
STS-2016Wu & Palmer: 0.2093 | Word2vec: 0.5860 | FastText: 0.2707 | SBERT: 0.7794 | RoBERTa: 0.8367 | ALBERT: 0.7956
STSB    Wu & Palmer: 0.2428 | Word2vec: 0.6274 | FastText: 0.2739 | SBERT: 0.8407 | RoBERTa: 0.9438 | ALBERT: 0.8448
SICK-R  Wu & Palmer: 0.4445 | Word2vec: 0.7006 | FastText: 0.4196 | SBERT: 0.8392 | RoBERTa: 0.8438 | ALBERT: 0.8088
MRPC    Wu & Palmer: 0.1759 | W